In [2]:
import pandas as pd
import numpy as np
import openpyxl
# !pip install openpyxl

In [3]:
## read excel data /Users/mingruichen/Documents/Spring-2025/CSCI1470/DL-Marriage-Prediction/Data/data_2013_to_2023.xlsx
# set working directory
import os
data = pd.read_excel('/Users/anita/Documents/CS1470/DL-Marriage-Prediction/Data/2013-2023_update.xlsx')

# School Classification

In [4]:

school_cols = [col for col in data.columns if 'school' in col.lower()]

print(" Columns containing 'school':")
for col in school_cols:
    non_null = data[col].notnull().sum()
    unique = data[col].nunique()
    print(f"- {col:<45} | Non-null: {non_null:<4} | Unique: {unique}")


 Columns containing 'school':
- first_partner_undergraduate_school            | Non-null: 8445 | Unique: 1168
- first_partner_graduate_school                 | Non-null: 5551 | Unique: 1674
- first_partner_father_undergraduate_school     | Non-null: 443  | Unique: 43
- first_partner_father_graduate_school          | Non-null: 429  | Unique: 28
- first_partner_mother_undergraduate_school     | Non-null: 422  | Unique: 20
- first_partner_mother_graduate_school          | Non-null: 426  | Unique: 25
- second_partner_undergraduate_school           | Non-null: 8461 | Unique: 1371
- second_partner_graduate_school                | Non-null: 5070 | Unique: 1409
- second_partner_father_undergraduate_school    | Non-null: 424  | Unique: 22
- second_partner_father_graduate_school         | Non-null: 448  | Unique: 45
- second_partner_mother_undergraduate_school    | Non-null: 421  | Unique: 18
- second_partner_mother_graduate_school         | Non-null: 426  | Unique: 23


In [5]:

all_school_names = set()

for col in school_cols:
    uniques = data[col].dropna().unique()
    all_school_names.update([u.strip() for u in uniques if u.strip() != ""])

all_school_names = sorted(all_school_names)

#print(f"{len(all_school_names)} unique school names.\n")

#for name in all_school_names:
    #print(name)


In [6]:

data_demo = data[['first_partner_gender', 'first_partner_age','first_partner_undergraduate_school',
                  'second_partner_gender','second_partner_age','second_partner_undergraduate_school']].copy()



In [7]:
import pandas as pd
import re

# ========== 1. Ivy League ==========
ivy_league = {
    'harvard university', 'yale university', 'princeton university', 'columbia university',
    'university of pennsylvania', 'brown university', 'dartmouth college', 'cornell university'
}

# ========== 2. Top 50 Private (Non-Ivy) National Universities ==========
top_50_private = {
    'massachusetts institute of technology', 'stanford university', 'university of chicago',
    'california institute of technology', 'duke university', 'johns hopkins university',
    'northwestern university', 'vanderbilt university', 'rice university', 'washington university in st. louis',
    'university of notre dame', 'emory university', 'georgetown university', 'carnegie mellon university',
    'university of southern california', 'tufts university', 'new york university', 'boston college'
}

# ========== 3. Top 50 Liberal Arts Colleges ==========
top_50_liberal_arts = {
    'williams college', 'amherst college', 'swarthmore college', 'pomona college', 'wellesley college',
    'bowdoin college', 'carleton college', 'claremont mckenna college', 'middlebury college',
    'washington and lee university', 'smith college', 'grinnell college', 'haverford college',
    'davidson college', 'vassar college', 'hamilton college', 'colby college', 'bates college',
    'wesleyan university', 'barnard college'
}

# ========== 4. Top 30 Public Universities ==========
top_30_public = {
    'university of california los angeles', 'university of california berkeley', 'university of michigan',
    'university of north carolina chapel hill', 'university of virginia', 'university of florida',
    'university of california santa barbara', 'university of california san diego', 'university of california davis',
    'university of california irvine', 'university of wisconsin madison', 'university of illinois urbana champaign',
    'university of texas austin', 'georgia institute of technology', 'university of georgia',
    'ohio state university', 'university of washington', 'purdue university', 'university of maryland college park',
    'university of pittsburgh'
}

# ========== 5. Alias Mapping ==========
alias_map = {
    # Ivy League
    'brown': 'brown university',
    'columbia': 'columbia university',
    'princeton': 'princeton university',
    'harvard': 'harvard university',
    'yale': 'yale university',
    'dartmouth': 'dartmouth college',
    'cornell': 'cornell university',
    # National Universities
    'stanford': 'stanford university',
    'mit': 'massachusetts institute of technology',
    'caltech': 'california institute of technology',
    'duke': 'duke university',
    'northwestern': 'northwestern university',
    'usc': 'university of southern california',
    'cmu': 'carnegie mellon university',
    'georgetown': 'georgetown university',
    'vanderbilt': 'vanderbilt university',
    'tufts': 'tufts university',
    'nyu': 'new york university',
    'n.y.u.': 'new york university',
    'washington university in st louis': 'washington university in st. louis',
    'notre dame': 'university of notre dame',
    # Liberal Arts
    'williams': 'williams college',
    'amherst': 'amherst college',
    'swarthmore': 'swarthmore college',
    'pomona': 'pomona college',
    'wellesley': 'wellesley college',
    'bowdoin': 'bowdoin college',
    'carleton': 'carleton college',
    'claremont mckenna': 'claremont mckenna college',
    'middlebury': 'middlebury college',
    'grinnell': 'grinnell college',
    'smith': 'smith college',
    'vassar': 'vassar college',
    'haverford': 'haverford college',
    'wesleyan': 'wesleyan university',
    'barnard': 'barnard college',
    # UC系统
    'uc berkeley': 'university of california berkeley',
    'ucla': 'university of california los angeles',
    'uc davis': 'university of california davis',
    'ucsd': 'university of california san diego',
    'uc irvine': 'university of california irvine',
    # Others
    'unc': 'university of north carolina chapel hill',
    'gpt-not found': ''
}


# ========== 6. 标准化函数 ==========
def normalize_school_name(name):
    if pd.isna(name):
        return ""
    name = name.strip().lower()
    name = name.replace('.', '').replace(',', '')  # 去掉小数点和逗号
    if name in {'n/a', 'not provided', 'not mentioned', 'gpt-not found', ''}:
        return ""  # 归为 No College Mentioned
    return alias_map.get(name, name)


# ========== 7. 分类函数 ==========
def classify_school_combined(school_entry):
    if pd.isna(school_entry) or str(school_entry).strip().lower() in {'n/a', 'not provided', 'not mentioned', 'gpt-not found', ''}:
        return "No College Mentioned"

    schools = re.split(r',| and |;', school_entry)
    categories = []
    
    for raw in schools:
        school = normalize_school_name(raw.strip())

        # Step 1: exact match
        if school in ivy_league:
            categories.append("Ivy League")
        elif school in top_50_private or school in top_50_liberal_arts:
            categories.append("Top 50 Private (Non-Ivy)")
        elif school in top_30_public:
            categories.append("Top 30 Public")
        else:
            # Step 2: fuzzy in match
            if any(iv in school for iv in ivy_league):
                categories.append("Ivy League")
            elif any(tp in school for tp in top_50_private.union(top_50_liberal_arts)):
                categories.append("Top 50 Private (Non-Ivy)")
            elif any(pub in school for pub in top_30_public):
                categories.append("Top 30 Public")
            else:
                categories.append("Other Colleges")

    # 返回最高优先级
    if "Ivy League" in categories:
        return "Ivy League"
    elif "Top 50 Private (Non-Ivy)" in categories:
        return "Top 50 Private (Non-Ivy)"
    elif "Top 30 Public" in categories:
        return "Top 30 Public"
    elif "Other Colleges" in categories:
        return "Other Colleges"
    else:
        return "No College Mentioned"


In [8]:
data_demo['first_partner_school_category'] = data_demo['first_partner_undergraduate_school'].apply(classify_school_combined)
data_demo['second_partner_school_category'] = data_demo['second_partner_undergraduate_school'].apply(classify_school_combined)


In [9]:
data_demo

,first_partner_gender,first_partner_age,first_partner_undergraduate_school,second_partner_gender,second_partner_age,second_partner_undergraduate_school,first_partner_school_category,second_partner_school_category
0,Female,31,Tufts,Male,32,Bowdoin College,Top 50 Private (Non-Ivy),Top 50 Private (Non-Ivy)
1,Female,30,Yale,NaN,30,Yale,Ivy League,Ivy League
2,Female,28,George Washington University,Male,35,Washington University in St. Louis,Other Colleges,Top 50 Private (Non-Ivy)
3,Female,34,University of Florida,Male,43,University of Miami,Top 30 Public,Other Colleges
4,Female,27,Boston University,Male,33,University of Pennsylvania,Other Colleges,Ivy League
...,...,...,...,...,...,...,...,...
9154,Female,30,Wagner College,Male,31,Wagner College,Other Colleges,Other Colleges
9155,Female,42,Learning Institute of Beauty Sciences,Female,50,Bowling Green State University,Other Colleges,Other Colleges
9156,Male,31,Georgetown,Female,33,Northwestern,Top 50 Private (Non-Ivy),Top 50 Private (Non-Ivy)
9157,Female,46,Michigan State University,Male,42,Lane College,Other Colleges,Other Colleges


In [10]:
# 抽样 50条
sample_check = data_demo[['first_partner_undergraduate_school', 'first_partner_school_category']].sample(50, random_state=42)

print(sample_check.to_string(index=False))


     first_partner_undergraduate_school first_partner_school_category
            City University of New York                Other Colleges
                       Oberlin College;                Other Colleges
               Miami University in Ohio                Other Colleges
               Parsons School of Design                Other Colleges
                        Rice University      Top 50 Private (Non-Ivy)
                   Villanova University                Other Colleges
                                    NaN          No College Mentioned
                                 N.Y.U.      Top 50 Private (Non-Ivy)
                                  Tufts      Top 50 Private (Non-Ivy)
                Old Dominion University                Other Colleges
                        Yale University                    Ivy League
                  Georgetown University      Top 50 Private (Non-Ivy)
                    New York University      Top 50 Private (Non-Ivy)
                    

In [11]:
"""
for cat in ['Ivy League', 'Top 50 Private (Non-Ivy)', 'Top 30 Public', 'Other Colleges', 'No College Mentioned']:
    subset = data_demo[data_demo['first_partner_school_category'] == cat]
    print(f"\nTop 10 schools in category '{cat}':")
    print(subset['first_partner_undergraduate_school'].value_counts().head(10))

"""


'\nfor cat in [\'Ivy League\', \'Top 50 Private (Non-Ivy)\', \'Top 30 Public\', \'Other Colleges\', \'No College Mentioned\']:\n    subset = data_demo[data_demo[\'first_partner_school_category\'] == cat]\n    print(f"\nTop 10 schools in category \'{cat}\':")\n    print(subset[\'first_partner_undergraduate_school\'].value_counts().head(10))\n\n'

# occupation classification

In [12]:
occupation_cols = [col for col in data.columns if 'occupation' in col.lower()]

print(" Columns containing 'occupation':")
for col in occupation_cols:
    non_null = data[col].notnull().sum()
    unique = data[col].nunique()
    print(f"- {col:<45} | Non-null: {non_null:<4} | Unique: {unique}")


 Columns containing 'occupation':
- first_partner_occupation                      | Non-null: 8767 | Unique: 5781
- first_partner_father_occupation               | Non-null: 5740 | Unique: 4127
- first_partner_mother_occupation               | Non-null: 4959 | Unique: 3356
- second_partner_occupation                     | Non-null: 8944 | Unique: 6152
- second_partner_father_occupation              | Non-null: 5876 | Unique: 4140
- second_partner_mother_occupation              | Non-null: 4905 | Unique: 3205


In [13]:

all_occupation_names = set()

for col in occupation_cols:
    uniques = data[col].dropna().unique()
    all_occupation_names.update([u.strip() for u in uniques if u.strip() != ""])

all_occupation_names = sorted(all_occupation_names)

#print(f"{len(all_occupation_names)} occupations .\n")

#for name in all_occupation_names:
    #print(name)


In [14]:

# ============================
# Step 1. 定义分类函数
# ============================

def classify_job_level(occupation):
    if pd.isna(occupation) or str(occupation).strip() == "":
        return "Other"
    
    occ = str(occupation).lower()
    if any(keyword in occ for keyword in ['chief', 'ceo', 'cfo', 'coo', 'president', 'founder', 'managing director']):
        return 'Executive'
    elif any(keyword in occ for keyword in ['senior', 'director', 'partner', 'lead', 'principal', 'vice president']):
        return 'Senior'
    elif any(keyword in occ for keyword in ['manager', 'engineer', 'specialist', 'architect', 'designer', 'scientist', 'teacher', 'professor']):
        return 'Mid'
    elif any(keyword in occ for keyword in ['assistant', 'associate', 'intern', 'analyst', 'aide', 'clerk', 'coordinator', 'technician']):
        return 'Entry'
    else:
        return 'Other'

def classify_job_field(occupation):
    if pd.isna(occupation) or str(occupation).strip() == "":
        return "Other/Unknown"
    
    occ = str(occupation).lower()
    if any(keyword in occ for keyword in ['finance', 'banker', 'accountant', 'financial', 'investment', 'analyst', 'trader']):
        return 'Finance/Business'
    elif any(keyword in occ for keyword in ['lawyer', 'attorney', 'legal', 'counsel', 'paralegal', 'judge']):
        return 'Law'
    elif any(keyword in occ for keyword in ['teacher', 'professor', 'instructor', 'educator', 'academic']):
        return 'Education'
    elif any(keyword in occ for keyword in ['physician', 'doctor', 'nurse', 'therapist', 'psychiatrist', 'surgeon', 'pharmacist']):
        return 'Healthcare'
    elif any(keyword in occ for keyword in ['engineer', 'developer', 'programmer', 'scientist', 'data scientist']):
        return 'Engineering/Tech'
    elif any(keyword in occ for keyword in ['artist', 'designer', 'photographer', 'actor', 'musician', 'editor', 'writer', 'journalist']):
        return 'Arts/Media'
    elif any(keyword in occ for keyword in ['officer', 'agent', 'politician', 'senator', 'mayor', 'diplomat']):
        return 'Government/Public Service'
    elif any(keyword in occ for keyword in ['marketing', 'sales', 'advertiser', 'representative', 'merchandiser']):
        return 'Sales/Marketing'
    elif any(keyword in occ for keyword in ['realtor', 'real estate', 'builder', 'contractor']):
        return 'Real Estate/Construction'
    elif any(keyword in occ for keyword in ['consultant', 'adviser', 'strategist']):
        return 'Consulting'
    else:
        return 'Other/Unknown'

def detect_retired(occupation):
    if pd.isna(occupation) or str(occupation).strip() == "":
        return 0
    
    occ = str(occupation).lower()
    if 'retired' in occ:
        return 1
    else:
        return 0

# ============================
# Step 2. 在all_occupation_names上分类，生成 mapping
# ============================

unique_occupations_df = pd.DataFrame({'occupation': all_occupation_names})

unique_occupations_df['job_level'] = unique_occupations_df['occupation'].apply(classify_job_level)
unique_occupations_df['job_field'] = unique_occupations_df['occupation'].apply(classify_job_field)
unique_occupations_df['is_retired'] = unique_occupations_df['occupation'].apply(detect_retired)

occupation_mapping = unique_occupations_df.set_index('occupation')[['job_level', 'job_field', 'is_retired']].to_dict(orient='index')

# ============================
# Step 3. 应用回原 dataframe data
# ============================

for col in occupation_cols:
    data[f'{col}_level'] = data[col].apply(lambda x: occupation_mapping.get(x.strip(), {'job_level': 'Other'})['job_level'] if pd.notna(x) and x.strip() != "" else 'Other')
    data[f'{col}_field'] = data[col].apply(lambda x: occupation_mapping.get(x.strip(), {'job_field': 'Other/Unknown'})['job_field'] if pd.notna(x) and x.strip() != "" else 'Other/Unknown')
    data[f'{col}_retired'] = data[col].apply(lambda x: occupation_mapping.get(x.strip(), {'is_retired': 0})['is_retired'] if pd.notna(x) and x.strip() != "" else 0)



print(data[[c for c in data.columns if any(key in c for key in ['occupation', 'level', 'field', 'retired'])]].head())


              first_partner_occupation  \
0               Architectural designer   
1              Director for operations   
2               Independent Consultant   
3                       Vice President   
4  Owner of an interior design company   

            first_partner_father_occupation  \
0  Surgical oncologist and medical director   
1         Leadership development consultant   
2                                       NaN   
3                                       NaN   
4                Retired marketing director   

                first_partner_mother_occupation  \
0  Adjunct assistant professor of public health   
1   Retired family and consumer science teacher   
2                                           NaN   
3                                           NaN   
4                                           NaN   

                           second_partner_occupation  \
0                                                NaN   
1                               Founder of Fa

# how_first_met classification

In [15]:
# how_first_met

#data['how_first_met']